# Notebook 03 – Statistical Validation

Validate whether relationships between user context (personality, mood, persona, device, shopping behaviour) and UI preferences are statistically significant.

Only statistically supported relationships are forwarded to later notebooks.


## Expected Outputs
- `reports/Statistical_Validation/tables/statistical_results.xlsx` (all tests retained)
- `reports/Statistical_Validation/tables/strong_evidence.xlsx`
- `reports/Statistical_Validation/tables/moderate_evidence.xlsx`
- `reports/Statistical_Validation/tables/exploratory_evidence.xlsx`
- `reports/Statistical_Validation/tables/top_evidence.xlsx`
- `reports/Statistical_Validation/statistical_summary.md`
- Heatmaps and figures under `reports/Statistical_Validation/`

> **Run the setup cell below first.** All relationships are kept; evidence tiers classify rather than discard results.


In [1]:
# --- Standard library ---
import logging
import sys
from pathlib import Path

# --- Third-party ---
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)

    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate

    raise FileNotFoundError(
        "Could not find project root containing src/config.py."
    )


_bootstrap_project()

# --- Project imports ---
from src.preprocessing.columns import (
    ALL_UI_COLUMNS,
    BIG_FIVE_LEVEL_COLUMNS,
    CONTEXT_FEATURE_COLUMNS,
    DESKTOP_UI_COLUMNS,
    GLOBAL_UI_COLUMNS,
    MOBILE_UI_COLUMNS,
    get_statistical_predictor_columns,
)
from src.statistics.reporting import generate_statistical_summary
from src.statistics.evidence import (
    export_statistical_outputs,
    summarize_evidence_statistics,
)
from src.statistics.validator import run_pairwise_validation
from src.utils.notebook import setup_notebook
from src.visualization.statistical_plots import (
    plot_association_heatmap,
    plot_top_relationships,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.figsize"] = (10, 6)

PATHS, REPORTS = setup_notebook("Statistical_Validation")
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures"
HEATMAPS = REPORTS / "heatmaps"
for folder in (TABLES, FIGURES, HEATMAPS):
    folder.mkdir(parents=True, exist_ok=True)

dataset_path = PATHS.data_processed / "clean_dataset.csv"
df = pd.read_csv(dataset_path)
logger.info("Loaded clean dataset: %s rows, %s columns", *df.shape)

# Context (mood, persona, device) + Big Five levels — 8 predictors × 41 UI elements
predictors = [
    column for column in get_statistical_predictor_columns() if column in df.columns
]
ui_elements = [column for column in ALL_UI_COLUMNS if column in df.columns]

print("Context predictors:", [c for c in predictors if c in CONTEXT_FEATURE_COLUMNS])
print("Big Five levels:", [c for c in predictors if c in BIG_FIVE_LEVEL_COLUMNS])
print(f"Global UI: {len(GLOBAL_UI_COLUMNS)} | Desktop: {len(DESKTOP_UI_COLUMNS)} | Mobile: {len(MOBILE_UI_COLUMNS)}")
print(f"Predictors: {len(predictors)} | UI elements: {len(ui_elements)} | Total pairs: {len(predictors) * len(ui_elements)}")
display(df[predictors + ui_elements[:3]].head())




INFO: Loaded clean dataset: 224 rows, 79 columns


Context predictors: ['primary_persona', 'current_mood', 'primary_device']
Big Five levels: ['Extraversion_Level', 'Agreeableness_Level', 'Conscientiousness_Level', 'Neuroticism_Level', 'Openness_Level']
Global UI: 14 | Desktop: 13 | Mobile: 14
Predictors: 8 | UI elements: 41 | Total pairs: 328


,primary_persona,current_mood,primary_device,Extraversion_Level,Agreeableness_Level,Conscientiousness_Level,Neuroticism_Level,Openness_Level,font_style_pref,font_size_pref,color_theme_pref
0,The Impulsive Buyer (I make quick decisions ba...,Neutral,Smartphone,Medium,Medium,Medium,Medium,Medium,2. Classic Serif (Traditional fonts with decor...,"2. Medium (14-16px - Standard size, balanced)","Warm Earthy (Browns, beiges, warm oranges - na..."
1,The Loyal Customer (I stick with brands and st...,Happy,Smartphone,Medium,High,Medium,Low,Low,"3. Rounded Friendly (Soft, approachable fonts ...",4. Extra Large (18+px - Maximum readability),"Minimalist Black & White (Clean, high contrast..."
2,The Browser (I enjoy browsing and discovering ...,Stressed,Smartphone,High,Medium,High,High,High,2. Classic Serif (Traditional fonts with decor...,4. Extra Large (18+px - Maximum readability),Cool Blues (Blues and grays - calm and trustwo...
3,"The Minimalist (I want simple, efficient shopp...",Happy,Smartphone,High,Medium,Medium,Medium,Medium,"3. Rounded Friendly (Soft, approachable fonts ...","1. Small (12-14px - Compact, more content visi...","Minimalist Black & White (Clean, high contrast..."
4,The Researcher (I thoroughly research products...,Bored,Smartphone,Low,High,High,Low,Low,"3. Rounded Friendly (Soft, approachable fonts ...",4. Extra Large (18+px - Maximum readability),"Vibrant Bold (Bright, energetic colors - excit..."


## Run Chi-Square Validation


In [2]:
results = run_pairwise_validation(df, predictors, ui_elements)
display(results.head(10))


,Predictor,UI_Element,ChiSquare,DOF,P_Value,Cramers_V,Effect_Size,Sample_Size,Adjusted_P,Significant,Significant_Nominal,Adjusted_P_Per_Predictor,Significant_Per_Predictor
0,current_mood,button_style_pref,68.504322,35,0.000604,0.247315,Medium,224,0.198035,False,True,0.024754,True
1,current_mood,mobile_sticky_header,14.540557,7,0.042361,0.254781,Medium,224,0.605783,False,True,0.434201,False
2,primary_device,button_style_pref,12.181644,5,0.032382,0.233200,Medium,224,0.605783,False,True,0.437018,False
3,current_mood,desktop_category_display,33.802117,21,0.038046,0.224278,Medium,224,0.605783,False,True,0.434201,False
4,current_mood,mobile_product_card,44.354319,28,0.025620,0.222492,Medium,224,0.605783,False,True,0.434201,False
5,current_mood,desktop_persistent_filters,11.056323,7,0.136181,0.222168,Medium,224,0.737642,False,False,0.620380,False
6,primary_persona,recommendation_type,32.184152,15,0.006077,0.218845,Medium,224,0.605783,False,True,0.167344,False
7,current_mood,mobile_filter_location,21.400994,14,0.091789,0.218564,Medium,224,0.635103,False,False,0.545182,False
8,current_mood,desktop_search_visibility,21.288008,14,0.094476,0.217986,Medium,224,0.635103,False,False,0.545182,False
9,Neuroticism_Level,button_style_pref,21.270113,10,0.019287,0.217894,Medium,224,0.605783,False,True,0.395380,False


## Export Full Results and Evidence Levels


In [3]:
export_paths = export_statistical_outputs(results, TABLES, top_n=30)
results = summarize_evidence_statistics(results, top_n=30)["enriched_results"]

print("Exported files:")
for name, file_path in export_paths.items():
    print(f"- {name}: {file_path}")

display(results[["Predictor", "UI_Element", "Raw_P", "Adjusted_P", "Adjusted_P_Per_Predictor", "Cramers_V", "Evidence_Score", "Evidence_Level"]].head(10))


INFO: Exported statistical outputs to /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables


Exported files:
- csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/exploratory_evidence.csv
- xlsx: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/exploratory_evidence.xlsx
- top_evidence_xlsx: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/top_evidence.xlsx
- top30_cramers_v_csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/top30_cramers_v.csv
- top30_evidence_score_csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/top30_evidence_score.csv


,Predictor,UI_Element,Raw_P,Adjusted_P,Adjusted_P_Per_Predictor,Cramers_V,Evidence_Score,Evidence_Level
0,current_mood,button_style_pref,0.000604,0.198035,0.024754,0.247315,1.000000,Moderate Statistical Evidence
1,current_mood,mobile_sticky_header,0.042361,0.605783,0.434201,0.254781,0.870490,Moderate Statistical Evidence
2,primary_device,button_style_pref,0.032382,0.605783,0.437018,0.233200,0.843000,Moderate Statistical Evidence
3,current_mood,desktop_category_display,0.038046,0.605783,0.434201,0.224278,0.829362,Moderate Statistical Evidence
4,current_mood,mobile_product_card,0.025620,0.605783,0.434201,0.222492,0.831291,Moderate Statistical Evidence
5,current_mood,desktop_persistent_filters,0.136181,0.737642,0.620380,0.222168,0.734566,Not Classified
6,primary_persona,recommendation_type,0.006077,0.605783,0.167344,0.218845,0.914671,Moderate Statistical Evidence
7,current_mood,mobile_filter_location,0.091789,0.635103,0.545182,0.218564,0.768315,Exploratory Evidence
8,current_mood,desktop_search_visibility,0.094476,0.635103,0.545182,0.217986,0.766549,Exploratory Evidence
9,Neuroticism_Level,button_style_pref,0.019287,0.605783,0.395380,0.217894,0.838976,Moderate Statistical Evidence


## Evidence Level Counts


In [4]:
print(f"Strong statistical evidence: {results['Is_Strong_Evidence'].sum()}")
print(f"Moderate statistical evidence: {results['Is_Moderate_Evidence'].sum()}")
print(f"Exploratory evidence: {results['Is_Exploratory_Evidence'].sum()}")
print(f"Nominally significant (Raw p < 0.05): {results['Significant_Nominal'].sum()}")
print(f"Global FDR significant: {results['Significant'].sum()}")
print(f"Per-predictor FDR significant: {results['Significant_Per_Predictor'].sum()}")

display(results[results["Is_Moderate_Evidence"]].head(10))


Strong statistical evidence: 0
Moderate statistical evidence: 12
Exploratory evidence: 50
Nominally significant (Raw p < 0.05): 25
Global FDR significant: 0
Per-predictor FDR significant: 1


,Predictor,UI_Element,ChiSquare,DOF,Raw_P,Adjusted_P,Adjusted_P_Per_Predictor,Cramers_V,Effect_Size,Sample_Size,Significant,Significant_Nominal,Significant_Per_Predictor,Evidence_Score,Evidence_Level,Is_Strong_Evidence,Is_Moderate_Evidence,Is_Exploratory_Evidence,P_Value
0,current_mood,button_style_pref,68.504322,35,0.000604,0.198035,0.024754,0.247315,Medium,224,False,True,True,1.000000,Moderate Statistical Evidence,False,True,True,0.000604
1,current_mood,mobile_sticky_header,14.540557,7,0.042361,0.605783,0.434201,0.254781,Medium,224,False,True,False,0.870490,Moderate Statistical Evidence,False,True,True,0.042361
2,primary_device,button_style_pref,12.181644,5,0.032382,0.605783,0.437018,0.233200,Medium,224,False,True,False,0.843000,Moderate Statistical Evidence,False,True,True,0.032382
3,current_mood,desktop_category_display,33.802117,21,0.038046,0.605783,0.434201,0.224278,Medium,224,False,True,False,0.829362,Moderate Statistical Evidence,False,True,True,0.038046
4,current_mood,mobile_product_card,44.354319,28,0.025620,0.605783,0.434201,0.222492,Medium,224,False,True,False,0.831291,Moderate Statistical Evidence,False,True,True,0.025620
6,primary_persona,recommendation_type,32.184152,15,0.006077,0.605783,0.167344,0.218845,Medium,224,False,True,False,0.914671,Moderate Statistical Evidence,False,True,True,0.006077
9,Neuroticism_Level,button_style_pref,21.270113,10,0.019287,0.605783,0.395380,0.217894,Medium,224,False,True,False,0.838976,Moderate Statistical Evidence,False,True,True,0.019287
10,primary_device,social_proof_display,10.561835,4,0.031956,0.605783,0.437018,0.217143,Medium,224,False,True,False,0.820691,Moderate Statistical Evidence,False,True,True,0.031956
11,primary_persona,mobile_price_display,21.052609,10,0.020730,0.605783,0.283306,0.216777,Medium,224,False,True,False,0.871135,Moderate Statistical Evidence,False,True,True,0.020730
14,Agreeableness_Level,button_style_pref,19.625733,10,0.032998,0.605783,0.636144,0.209302,Medium,224,False,True,False,0.748527,Moderate Statistical Evidence,False,True,True,0.032998


## Generate Summary Report


In [5]:
summary_path = generate_statistical_summary(
    results,
    REPORTS / "statistical_summary.md",
)
print(f"Summary saved to: {summary_path}")


Summary saved to: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/statistical_summary.md


## Visualizations


In [6]:
plot_association_heatmap(
    results,
    "Adjusted_P",
    HEATMAPS / "adjusted_p_heatmap.png",
    title="Adjusted P-Values (FDR)",
    cmap="YlOrRd_r",
    fmt=".3f",
)
plot_association_heatmap(
    results,
    "Cramers_V",
    HEATMAPS / "cramers_v_heatmap.png",
    title="Cramér's V Effect Sizes",
    cmap="Blues",
    fmt=".2f",
)
plot_top_relationships(
    results,
    FIGURES / "top_30_relationships.png",
    top_n=30,
)


PosixPath('/Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/figures/top_30_relationships.png')

## Final Summary


In [7]:
summary = summarize_evidence_statistics(results, top_n=30)
strongest_v = summary["strongest_cramers_v"]
strongest_score = summary["strongest_evidence_score"]

print(f"Total Tests: {summary['total_tests']}")
print(f"Nominally Significant (Raw p < 0.05): {summary['significant_nominal']}")
print(f"Significant after Global FDR: {summary['significant_global_fdr']}")
print(f"Significant after Predictor-level FDR: {summary['significant_per_predictor_fdr']}")
print(f"Medium Effect Sizes: {summary['medium_effect_sizes']}")
print(f"Large Effect Sizes: {summary['large_effect_sizes']}")
print(f"Strong Evidence: {summary['strong_evidence_count']}")
print(f"Moderate Evidence: {summary['moderate_evidence_count']}")
print(f"Exploratory Evidence: {summary['exploratory_evidence_count']}")
print(
    "Strongest by Cramer's V: "
    f"{strongest_v['Predictor']} × {strongest_v['UI_Element']} "
    f"(V={strongest_v['Cramers_V']:.3f}, Raw p={strongest_v['Raw_P']:.4f})"
)
print(
    "Strongest by Evidence Score: "
    f"{strongest_score['Predictor']} × {strongest_score['UI_Element']} "
    f"(Score={strongest_score['Evidence_Score']:.3f}, V={strongest_score['Cramers_V']:.3f})"
)
print(f"Average Cramer's V: {summary['average_cramers_v']:.3f}")
print(f"Average Evidence Score: {summary['average_evidence_score']:.3f}")
print("Export Locations:")
print(f"- {TABLES / 'statistical_results.xlsx'}")
print(f"- {TABLES / 'strong_evidence.xlsx'}")
print(f"- {TABLES / 'moderate_evidence.xlsx'}")
print(f"- {TABLES / 'exploratory_evidence.xlsx'}")
print(f"- {TABLES / 'top_evidence.xlsx'}")
print(f"- {REPORTS / 'statistical_summary.md'}")


Total Tests: 328
Nominally Significant (Raw p < 0.05): 25
Significant after Global FDR: 0
Significant after Predictor-level FDR: 1
Medium Effect Sizes: 21
Large Effect Sizes: 0
Strong Evidence: 0
Moderate Evidence: 12
Exploratory Evidence: 50
Strongest by Cramer's V: current_mood × mobile_sticky_header (V=0.255, Raw p=0.0424)
Strongest by Evidence Score: current_mood × button_style_pref (Score=1.000, V=0.247)
Average Cramer's V: 0.126
Average Evidence Score: 0.441
Export Locations:
- /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/statistical_results.xlsx
- /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/strong_evidence.xlsx
- /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Validation/tables/moderate_evidence.xlsx
- /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/Statistical_Valida